# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and fields, referencing by their `@id`.

In [ ]:
# List all record sets by @id and name
record_sets_info = []
for record_set in dataset.record_sets:
    record_sets_info.append({'@id': record_set['@id'], 'name': record_set.get('name', '')})
if not record_sets_info:
    print("No record sets found in the metadata. Attempting to load main/only record set.")
else:
    for info in record_sets_info:
        print(f"Record set: {info['name']} (@id: {info['@id']})")

In [ ]:
# If record sets are present, print the fields for each set by @id
for record_set in dataset.record_sets:
    print(f"Record Set @id: {record_set['@id']}, name: {record_set.get('name', '')}")
    fields = record_set.get('field', [])
    # If only one field, convert to list
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        field_obj = field if isinstance(field, dict) else dataset.fields_by_id.get(field)
        if field_obj:
            print(f"  Field: {field_obj.get('name', '')} (@id: {field_obj.get('@id', field)})")
        else:
            print(f"  Field @id: {field}")

## 3. Data Extraction
Load data from available record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Find all available record sets (by @id); use the first one as default
if len(dataset.record_sets) > 0:
    record_sets_ids = [rs['@id'] for rs in dataset.record_sets]
    # Load all record_set data into dataframes
    dataframes = {}
    for record_set_id in record_sets_ids:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
    first_set_id = record_sets_ids[0]
    print(f"Columns in record set {first_set_id}:")
    print(dataframes[first_set_id].columns.tolist())
    print(dataframes[first_set_id].head())
else:
    # If no explicit record sets, try loading records generally
    records = list(dataset.records())
    df = pd.DataFrame(records)
    print("Loaded records (no explicit record set):")
    print(df.columns.tolist())
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section will include operations like removing outliers, transforming data distributions, or grouping data by attributes to prepare for further analysis.

In [ ]:
# Identify candidate numeric columns
import numpy as np
numeric_candidates = []
df_to_use = None
df_id_to_use = None
if len(dataset.record_sets) > 0:
    for rsid in dataframes:
        df = dataframes[rsid]
        for col in df.columns:
            # count as numeric if dtype is float or int or can be cast
            if np.issubdtype(df[col].dtype, np.number):
                numeric_candidates.append({'record_set_id': rsid, 'col': col})
        if len(numeric_candidates) > 0:
            df_to_use = df
            df_id_to_use = rsid
            break
else:
    # fallback
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_candidates.append({'record_set_id': None, 'col': col})
    df_to_use = df
    df_id_to_use = None

if len(numeric_candidates) == 0:
    print("No numeric fields found in records.")
else:
    # Use first candidate numeric
    numeric_field = numeric_candidates[0]['col']
    print(f"Selected numeric field: {numeric_field}\n")
    threshold = df_to_use[numeric_field].mean() if df_to_use[numeric_field].notnull().sum() > 0 else 1
    filtered_df = df_to_use[df_to_use[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Find any categorical string column
    group_field = None
    for col in df_to_use.columns:
        if df_to_use[col].dtype == object and col != numeric_field:
            group_field = col
            break
    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Example: histogram or scatterplot for selected numeric fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if df_to_use is not None and len(numeric_candidates) > 0:
    plt.figure(figsize=(8, 4))
    sns.histplot(df_to_use[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of "{numeric_field}"')
    plt.xlabel(numeric_field)
    plt.show()
    
    # If at least 2 numerics, show scatter
    if len(numeric_candidates) > 1:
        numeric_field2 = numeric_candidates[1]['col']
        plt.figure(figsize=(7, 5))
        plt.scatter(df_to_use[numeric_field], df_to_use[numeric_field2], alpha=0.5)
        plt.xlabel(numeric_field)
        plt.ylabel(numeric_field2)
        plt.title(f'Scatterplot: {numeric_field} vs. {numeric_field2}')
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded and inspected metadata and data with `mlcroissant`.
- Identified available record sets and their fields via `@id`.
- Demonstrated basic EDA: filtered numeric records, normalized fields, group-wise summaries.
- Visualized primary numeric distributions.
- For further biological interpretation, see the original publication and Croissant schema.